In [1]:
from collections.abc import Callable
from typing import Any, ClassVar, Literal

import equinox as eqx
import jax
import jax.numpy as jnp
import numpy as np
import xarray as xr
from clawpack import pyclaw
from context_flux_no.simulations.pdesolve import pdesolve_pyclaw
from jaxtyping import Array, Float, PRNGKeyArray
from tqdm import tqdm


jax.config.update("jax_enable_x64", True)


def riemann_cubic_1D(
    q_l: Float[np.ndarray, "num_eqns num_riemanns"],
    q_r: Float[np.ndarray, "num_eqns num_riemanns"],
    aux_l: Float[np.ndarray, "num_aux num_riemanns"],
    aux_r: Float[np.ndarray, "num_aux num_riemanns"],
    problem_data: dict[str, float],
) -> tuple[
    Float[np.ndarray, "num_eqns num_waves num_riemanns"],
    Float[np.ndarray, "num_waves num_riemanns"],
    Float[np.ndarray, "num_eqns num_riemanns"],
    Float[np.ndarray, "num_eqns num_riemanns"],
]:
    """
    num_riemanns correspond to number of grid cells
    For this problem, num_eqns=1 and num_waves=1.
    """

    a, b, c = problem_data["a"], problem_data["b"], problem_data["c"]

    dq: Float[np.ndarray, "num_eqns num_riemanns"] = q_r - q_l
    # Horner's rule for slightly faster polynomial evaluation
    f_r = ((a * q_r + b) * q_r) + c * q_r
    f_l = ((a * q_l + b) * q_l + c) * q_l
    s_default = (3 * a * q_l + 2 * b) * q_l + c

    s = np.where(np.abs(dq) > 1e-14, (f_r - f_l) / dq, s_default)
    wave = np.expand_dims(dq, axis=1)
    amdq = np.clip(s, max=0.0) * dq
    apdq = np.clip(s, min=0.0) * dq
    return wave, s, amdq, apdq


class CubicFlux1D(eqx.Module):
    n_dim: ClassVar[int] = 1
    n_eqns: ClassVar[int] = 1
    a: float = eqx.field(static=True)
    b: float = eqx.field(static=True)
    c: float = eqx.field(static=True)

    def __init__(self, a: float = 1.0, b: float = 1.0, c: float = 1.0):
        self.a = a
        self.b = b
        self.c = c

    @property
    def coeffs(self) -> dict[str, float]:
        return {"a": self.a, "b": self.b, "c": self.c}

    def solve_pyclaw(
        self,
        ic_factory: Callable[[Float[np.ndarray, " Nx"]], Float[np.ndarray, " Nx"]],
        x_span: tuple[float, float],
        Nx: int,
        t_span: tuple[float, float],
        Nt: int,
        bc: Literal["periodic"],
    ) -> tuple[
        Float[np.ndarray, "time dim x_grid"],
        Float[np.ndarray, " time"],
        Float[np.ndarray, " x_grid"],
    ]:
        solver = pyclaw.ClawSolver1D(riemann_cubic_1D)
        solver.limiters = pyclaw.limiters.tvd.MC
        solver.num_eqn = self.n_eqns
        solver.num_waves = 1
        solver.kernel_language = "Python"
        solver.cfl_desired = 0.5
        solver.cfl_max = 0.9
        solver.fwave = False

        problem_data = {"a": self.a, "b": self.b, "c": self.c}
        return pdesolve_pyclaw(
            solver, problem_data, ic_factory, x_span, Nx, t_span, Nt, bc
        )


In [2]:
def sample_pde(
    pde_factory: Callable[[Any], eqx.Module],
    *,
    seed: int = 0,
    **kwargs: dict[str, tuple[float, float]],
) -> eqx.Module:
    keys = jax.random.split(jax.random.key(seed), len(kwargs))

    param_dict = dict()
    for rng_key, (param_name, sample_range) in zip(keys, kwargs.items()):
        param_dict[param_name] = float(
            jax.random.uniform(
                rng_key,
                minval=sample_range[0],
                maxval=sample_range[1],
            )
        )

    return pde_factory(**param_dict)

In [3]:
pde = sample_pde(CubicFlux1D, a=(-1.0, 1.0), b=(-1.0, 1.0), c=(-1.0, 1.0))

INFO:2025-09-17 16:48:10,503:jax._src.xla_bridge:749: Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory


2025-09-17 16:48:10,503 INFO CLAW: Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory


In [ ]:
from context_flux_no.gaussian_random_field import (
    GaussianCov,
    generate_circulant_embedding_method_1d,
)


u0_func = lambda x_: generate_circulant_embedding_method_1d(
    len(x_), x_[1] - x_[0], GaussianCov(0.05)
)
u, t, x = pde.solve_pyclaw(u0_func, (-1.0, 1.0), 256, (0.0, 0.1), 100, "periodic")

2025-09-17 16:54:06.510835: E external/xla/xla/service/slow_operation_alarm.cc:73] Constant folding an instruction is taking > 1s:

  %convert.144 = c128[] convert(%constant.4), metadata={op_name="jit(generate_circulant_embedding_method_1d)/jit(main)/convert_element_type" source_file="/home/jhko725/projects/CONTEXT_FLUX_NO/src/context_flux_no/gaussian_random_field.py" source_line=117}

This isn't necessarily a bug; constant-folding is inherently a trade-off between compilation time and speed at runtime. XLA has some guards that attempt to keep constant folding from taking too long, but fundamentally you'll always be able to come up with an input program that takes a long time.

If you'd like to file a bug, run with envvar XLA_FLAGS=--xla_dump_to=/tmp/foo and attach the results.
2025-09-17 16:54:06.948250: E external/xla/xla/service/slow_operation_alarm.cc:140] The operation took 1.437548027s
Constant folding an instruction is taking > 1s:

  %convert.144 = c128[] convert(%constant.4), 

2025-09-17 16:56:11,351 INFO CLAW: Solution 0 computed for time t=0.000000
2025-09-17 16:56:16,059 INFO CLAW: Solution 1 computed for time t=0.000708
2025-09-17 16:56:20,717 INFO CLAW: Solution 2 computed for time t=0.000741
2025-09-17 16:56:25,276 INFO CLAW: Solution 3 computed for time t=0.000748
2025-09-17 16:56:29,886 INFO CLAW: Solution 4 computed for time t=0.000750
2025-09-17 16:56:34,445 INFO CLAW: Solution 5 computed for time t=0.000753
2025-09-17 16:56:39,177 INFO CLAW: Solution 6 computed for time t=0.000755
2025-09-17 16:56:43,623 INFO CLAW: Solution 7 computed for time t=0.000757
2025-09-17 16:56:48,463 INFO CLAW: Solution 8 computed for time t=0.000760
2025-09-17 16:56:52,872 INFO CLAW: Solution 9 computed for time t=0.000762
2025-09-17 16:56:57,673 INFO CLAW: Solution 10 computed for time t=0.000764
2025-09-17 16:57:02,083 INFO CLAW: Solution 11 computed for time t=0.000767
2025-09-17 16:57:06,861 INFO CLAW: Solution 12 computed for time t=0.000769
2025-09-17 16:57:11,41

/tmp/ipykernel_740/162995243.py:43: RuntimeWarning: divide by zero encountered in divide
  s = np.where(np.abs(dq) > 1e-14, (f_r - f_l) / dq, s_default)


2025-09-17 17:00:59,892 INFO CLAW: Solution 62 computed for time t=0.001089
2025-09-17 17:01:05,091 INFO CLAW: Solution 63 computed for time t=0.001162
2025-09-17 17:01:10,323 INFO CLAW: Solution 64 computed for time t=0.001223
2025-09-17 17:01:15,585 INFO CLAW: Solution 65 computed for time t=0.001253


Exception: Maximum number of timesteps have been taken

In [ ]:
def sample_coefficients_uniform(
    key: PRNGKeyArray, coeff_range_dict: dict[str, tuple[float, float]]
):
    subkeys = jax.random.split(key, len(coeff_range_dict))
    return {
        name: float(
            jax.random.uniform(subkey, minval=coeff_range[0], maxval=coeff_range[1])
        )
        for subkey, (name, coeff_range) in zip(subkeys, coeff_range_dict.items())
    }


## TODO: Generalize to nd
def solution_to_dataarray(
    u: Float[Array, "time dim x_grid"],
    t: Float[Array, " time"],
    x: Float[Array, " x_grid"],
    coeffs: dict[str, float],
) -> xr.DataArray:
    return xr.DataArray(
        np.expand_dims(u, axis=0),
        coords={
            "t": t,
            "x": x,
            "dim": [
                "u",
            ],
            **{coeff: ("sample", [value]) for coeff, value in coeffs.items()},
        },
        dims=["sample", "t", "dim", "x"],
    )


In [8]:
def generate_dataset(
    n_samples: int,
    pde_factory: Callable[[Any], eqx.Module],
    initial_condition_fn: Callable[[Array, PRNGKeyArray], Array],
    coeff_range_dict: dict,
    x_span: tuple[float, float],
    Nx: int,
    t_span: tuple[float, float],
    Nt: int,
    bc: Literal["periodic"] = "periodic",
    seed: int = 0,
):
    keys = jax.random.split(jax.random.key(seed), n_samples)

    solutions = []
    for key in tqdm(keys):
        key_coeff, key_ic = jax.random.split(key)
        coeffs = sample_coefficients_uniform(key_coeff, coeff_range_dict)
        pde = pde_factory(**coeffs)

        sol = pde.solve_pyclaw(
            lambda u0: initial_condition_fn(u0, key_ic), x_span, Nx, t_span, Nt, bc
        )
        solutions.append(*sol, pde.coeffs)

    return xr.concat(solutions, "sample")

In [4]:
class TruncatedFourier1D(eqx.Module):
    an: Float[Array, " K"]
    bn: Float[Array, " K"]
    a0: Float[Array, ""] = eqx.field(default_factory=lambda: jnp.array(0.0))
    L: float = eqx.field(static=True, default=1.0)

    def __call__(self, x: Float[Array, " *shape"]):
        kx: Float[Array, "*shape K"] = jnp.expand_dims(x, axis=-1) * self.wavenumbers
        cos_kx, sin_kx = jnp.cos(kx), jnp.sin(kx)
        return self.a0 + jnp.mean(
            self.an * cos_kx + self.bn * sin_kx, axis=-1
        ) / jnp.sqrt(2)

    @property
    def num_modes(self) -> int:
        return len(self.an)

    @property
    def wavenumbers(self) -> Float[Array, " K"]:
        return jnp.arange(1, self.num_modes + 1) * 2 * jnp.pi / self.L

    @classmethod
    def with_uniform_rand_coeffs(
        cls,
        num_modes: int,
        L: float = 1.0,
        coeff_range: tuple[float, float] = (1, 1),
        offset_range: tuple[float, float] | None = None,
        *,
        key: PRNGKeyArray = jax.random.PRNGKey(0),
    ):
        key_coeff, key_offset = jax.random.split(key, 2)
        an_bn = jax.random.uniform(
            key_coeff,
            shape=(2, num_modes),
            minval=coeff_range[0],
            maxval=coeff_range[1],
        )
        if offset_range is None:
            a0 = jnp.array(0.0)
        else:
            a0 = jax.random.uniform(
                key_offset, minval=offset_range[0], maxval=offset_range[1]
            )
        return cls(*an_bn, a0, L)

In [ ]:
dataset = generate_dataset(
    n_samples=10,
    pde_factory=CubicFlux1D,
    initial_condition_fn=lambda u0, key: TruncatedFourier1D.with_uniform_rand_coeffs(
        num_modes=4, key=key
    )(u0),
    coeff_range_dict={"a": (-1.0, 1.0), "b": (-1.0, 1.0), "c": (-1.0, 1.0)},
    x_span=(-1, 1),
    Nx=256,
    t_span=(0, 0.1),
    Nt=100,
    seed=0,
)

  0%|          | 0/10 [00:00<?, ?it/s]/tmp/ipykernel_4179890/1382419846.py:19: UserWarning: A JAX array is being set as static! This can result in unexpected behavior and is usually a mistake to do.
  pde = pde_factory(**coeffs)


2025-09-17 16:39:55,859 INFO CLAW: Solution 0 computed for time t=0.000000
2025-09-17 16:40:10,560 INFO CLAW: Solution 1 computed for time t=0.001000
2025-09-17 16:40:35,076 INFO CLAW: Solution 2 computed for time t=0.001929
2025-09-17 16:40:58,219 INFO CLAW: Solution 3 computed for time t=0.001929
